In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


: 

In [ ]:
DATA_DIR = Path("../../data/model2_seniority/3.processed")
CSV_PATH = DATA_DIR / "cv_with_seniority_weak.csv"

df = pd.read_csv(CSV_PATH)

In [ ]:
print(df.shape)
df.head()


In [ ]:
# Etiquetas válidas
VALID_LABELS = ["Junior", "Mid", "Senior"]

# Aseguramos texto como string y sin NaN
df["cv_text"] = df["cv_text"].fillna("").astype(str)

# Filtrado por etiqueta débil de seniority
mask_labels = df["seniority_weak"].isin(VALID_LABELS)
df_train_seniority = df[mask_labels].copy()

# Eliminar duplicados por cv_id (por seguridad)
if "cv_id" in df_train_seniority.columns:
    df_train_seniority = df_train_seniority.drop_duplicates(subset="cv_id")

print("Tamaño df_train_seniority:", df_train_seniority.shape)
df_train_seniority[["cv_id", "seniority_weak"]].head()


In [ ]:
class_counts = df_train_seniority["seniority_weak"].value_counts()
class_ratio = df_train_seniority["seniority_weak"].value_counts(normalize=True)

print("Recuento por clase:")
print(class_counts)
print("\nProporción por clase:")
print(class_ratio)


In [ ]:
X = df_train_seniority["cv_text"].values
y = df_train_seniority["seniority_weak"].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Tamaño X_train:", len(X_train))
print("Tamaño X_test :", len(X_test))


In [ ]:
print("Train class distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts(normalize=True))


In [ ]:
from sklearn.utils import resample

# Pasamos el train a DataFrame para trabajar más cómodo
train_df = pd.DataFrame({
    "cv_text": X_train,
    "seniority_weak": y_train,
})

# Lo separamos por clase
df_junior = train_df[train_df["seniority_weak"] == "Junior"]
df_mid    = train_df[train_df["seniority_weak"] == "Mid"]
df_senior = train_df[train_df["seniority_weak"] == "Senior"]

print(len(df_junior), len(df_mid), len(df_senior))


In [ ]:
# Objetivo: mismo nº de ejemplos que Senior
n_target = len(df_senior)

df_junior_up = resample(
    df_junior,
    replace=True,
    n_samples=n_target,
    random_state=42,
)

df_mid_up = resample(
    df_mid,
    replace=True,
    n_samples=n_target,
    random_state=42,
)

# Juntamos todo y mezclamos
train_df_balanced = pd.concat([df_senior, df_mid_up, df_junior_up], axis=0)
train_df_balanced = train_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(train_df_balanced["seniority_weak"].value_counts())


In [ ]:
X_train_bal = train_df_balanced["cv_text"].values
y_train_bal = train_df_balanced["seniority_weak"].values


### Model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

text_clf_bal = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=5,
                max_df=0.8,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                n_jobs=-1,
                multi_class="auto",
            ),
        ),
    ]
)




In [ ]:
%%time
text_clf_bal.fit(X_train_bal, y_train_bal)

In [ ]:
y_pred_test_bal = text_clf_bal.predict(X_test)

print("=== Classification report (TEST) - train balanceado ===")
print(classification_report(y_test, y_pred_test_bal, digits=3))

labels = ["Junior", "Mid", "Senior"]
cm_bal = confusion_matrix(y_test, y_pred_test_bal, labels=labels)

cm_bal_df = pd.DataFrame(
    cm_bal,
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels],
)

print("=== Matriz de confusión (TEST) - balanceado ===")
print(cm_bal_df)

cm_bal_norm = cm_bal_df.div(cm_bal_df.sum(axis=1), axis=0)
print("\n=== Matriz de confusión normalizada (TEST) - balanceado ===")
print(cm_bal_norm.round(3))


### Guardar modelo

In [ ]:
from pathlib import Path
import joblib

MODEL_DIR = Path("../../models/seniority")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "seniority_from_cv_balanced_v1.pkl"

joblib.dump(text_clf_bal, MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")
